<a href="https://colab.research.google.com/github/rbevan-png/NLP_Text_to_SQL_Project/blob/main/T5SQL2SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Install Dependencies and Prepare Dataset

In this step, we firstly install necessary packages, and we also unzip the Spider dataset to train our model. Secondly, we preprocess spider to include the question, schema, and target query in the training and validation datasets.

In [1]:
pip install transformers datasets sentencepiece accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!unzip -o /content/drive/MyDrive/spider.zip -d spider

Archive:  /content/drive/MyDrive/spider.zip
  inflating: spider/spider/README.txt  
  inflating: spider/spider/database/academic/academic.sqlite  
  inflating: spider/spider/database/academic/schema.sql  
  inflating: spider/spider/database/activity_1/activity_1.sqlite  
  inflating: spider/spider/database/activity_1/schema.sql  
  inflating: spider/spider/database/aircraft/aircraft.sqlite  
  inflating: spider/spider/database/aircraft/schema.sql  
  inflating: spider/spider/database/allergy_1/allergy_1.sqlite  
  inflating: spider/spider/database/allergy_1/schema.sql  
  inflating: spider/spider/database/apartment_rentals/apartment_rentals.sqlite  
  inflating: spider/spider/database/apartment_rentals/schema.sql  
  inflating: spider/spider/database/architecture/architecture.sqlite  
  inflating: spider/spider/database/architecture/schema.sql  
  inflating: spider/spider/database/assets_maintenance/assets_maintenance.sqlite  
  inflating: spider/spider/database/assets_maintenance/sche

In [3]:
import os
import json
from datasets import Dataset

base_dir = "spider/spider"
train_file  = os.path.join(base_dir, "train_spider.json")
val_file = os.path.join(base_dir, "dev.json")
schema_file = os.path.join(base_dir, "tables.json")

def load_spider_json(path):
    with open(path, "r") as f:
        return json.load(f)

def load_schema(schema_path):
    with open(schema_path, "r") as f:
        tables = json.load(f)

    schema_dict = {}
    for table in tables:
        db_id         = table["db_id"]
        table_names   = table["table_names_original"]
        column_data   = table["column_names_original"]

        full_schema = []
        for ti, tbl in enumerate(table_names):
            cols = [col_name for (tbl_idx, col_name) in column_data
                    if tbl_idx == ti and col_name != "*"]
            if cols:
                full_schema.append(f"{tbl}({', '.join(cols)})")

        schema_dict[db_id] = " | ".join(full_schema)
    return schema_dict

def preprocess_spider(spider_data, schema_dict):
    examples = []
    for item in spider_data:
        db_id    = item["db_id"]
        question = item["question"]
        sql      = item["query"]
        schema   = schema_dict.get(db_id, "")

        prompt = (
            f"translate question to SQL: {question}"
            f" | schema: {schema}"
        )
        examples.append({"input": prompt, "target": sql})
    return Dataset.from_list(examples)

train_data = load_spider_json(train_file)
val_data = load_spider_json(val_file)
schema_dict = load_schema(schema_file)

train_dataset = preprocess_spider(train_data, schema_dict)
val_dataset = preprocess_spider(val_data, schema_dict)

print(train_dataset[0])
print(val_dataset[0])

{'input': 'translate question to SQL: How many heads of the departments are older than 56 ? | schema: department(Department_ID, Name, Creation, Ranking, Budget_in_Billions, Num_Employees) | head(head_ID, name, born_state, age) | management(department_ID, head_ID, temporary_acting)', 'target': 'SELECT count(*) FROM head WHERE age  >  56'}
{'input': 'translate question to SQL: How many singers do we have? | schema: stadium(Stadium_ID, Location, Name, Capacity, Highest, Lowest, Average) | singer(Singer_ID, Name, Country, Song_Name, Song_release_year, Age, Is_male) | concert(concert_ID, concert_Name, Theme, Stadium_ID, Year) | singer_in_concert(concert_ID, Singer_ID)', 'target': 'SELECT count(*) FROM singer'}


# 2. Model Training

In this section, we utilize google/t5-large-lm-adapt (https://huggingface.co/google/t5-large-lm-adapt) as the base model we are going to fine-tune. We train the model for 5 epochs using the Seq2SeqTrainer (https://huggingface.co/docs/transformers/v4.51.3/en/main_classes/trainer#transformers.Seq2SeqTrainer).

In [5]:
from transformers import T5Tokenizer, T5ForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments

model = T5ForConditionalGeneration.from_pretrained("google/t5-large-lm-adapt")
tokenizer = T5Tokenizer.from_pretrained("google/t5-large-lm-adapt")

def tokenize(batch):
    inputs = tokenizer(batch['input'], padding="max_length", truncation=True, max_length=128)
    targets = tokenizer(batch['target'], padding="max_length", truncation=True, max_length=128)
    inputs["labels"] = targets["input_ids"]
    return inputs

train_tokenized = train_dataset.map(tokenize, batched=True)
val_tokenized = val_dataset.map(tokenize, batched=True)

training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-sql",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    logging_dir="./logs",
    num_train_epochs=5,
    save_strategy="epoch"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer
)

trainer.train()


config.json:   0%|          | 0.00/656 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.11k [00:00<?, ?B/s]

...

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rishi-aniga (rishi-aniga-georgia-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss
500,1.366500
1000,0.106600
1500,0.081300
2000,0.067200
2500,0.059400
3000,0.051000
3500,0.047500
4000,0.041600


TrainOutput(global_step=4375, training_loss=0.21160811266217913, metrics={'train_runtime': 2004.3286, 'train_samples_per_second': 17.462, 'train_steps_per_second': 2.183, 'total_flos': 2.016671956992e+16, 'train_loss': 0.21160811266217913, 'epoch': 5.0})

In [7]:
inputs = tokenizer("translate question to SQL: How many singers older than 5 do we have?", return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}
outputs = model.generate(**inputs)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

SELECT count(*) FROM singers WHERE age > 5


# 3. Model Evaluation

We will evaluate the model on the Spider dataset using two metrics: exact match and execution match. Exact match measures how many queries are exactly correct after removing formatting differences, while execution match compares the results of the queries.

In [17]:
import os
import json
import sqlite3
import re
import torch
from tqdm import tqdm
from transformers import T5Tokenizer, T5ForConditionalGeneration

val_file = "spider/spider/dev.json"
db_dir = "spider/spider/database"
schema_file = "spider/spider/tables.json"
model_dir = "t5-sql/checkpoint-4375"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = T5Tokenizer.from_pretrained(model_dir)
model = T5ForConditionalGeneration.from_pretrained(model_dir).to(device)
model.eval()

with open(val_file) as f:
    val_data = json.load(f)

def load_schema(path):
    d = {}
    for tbl in json.load(open(path)):
        db_id = tbl["db_id"]
        names = tbl["table_names_original"]
        cols = tbl["column_names_original"]
        parts = []
        for ti, nm in enumerate(names):
            cs = [c for (i,c) in cols if i==ti and c!="*"]
            if cs:
                parts.append(f"{nm}({', '.join(cs)})")
        d[db_id] = " | ".join(parts)
    return d

schema_dict = load_schema(schema_file)

def make_prompt(question, db_id):
    return f"translate question to SQL: {question} | schema: {schema_dict[db_id]}"

def clean_sql(sql: str) -> str:
    sql = sql.strip().lower()
    sql = sql.replace(" ;", ";").replace(" ,", ",")
    sql = re.sub(r'"([^"]*)"', r"'\1'", sql)
    sql = re.sub(r'\s+', ' ', sql)
    return sql

def execute_sql(query, db_path):
    try:
        conn = sqlite3.connect(db_path)
        cur  = conn.cursor()
        cur.execute(query)
        res = cur.fetchall()
        conn.close()
        return res
    except:
        return None

total = 0
em_correct = 0
exec_total = 0
exec_matches = 0

em_mismatches = []
exec_mismatches = []

for item in tqdm(val_data):
    db_id = item["db_id"]
    question = item["question"]
    gold_sql = item["query"]

    prompt = make_prompt(question, db_id)
    toks   = tokenizer(prompt, return_tensors="pt",
                       truncation=True, padding=True, max_length=256).to(device)
    with torch.no_grad():
        out = model.generate(**toks, max_length=256)
    pred_sql = tokenizer.decode(out[0], skip_special_tokens=True).strip()

    n_gold = clean_sql(gold_sql)
    n_pred = clean_sql(pred_sql)
    is_em = (n_gold == n_pred)
    if not is_em and len(em_mismatches) < 5:
        em_mismatches.append({
            "question": question,
            "gold": gold_sql,
            "pred": pred_sql,
            "normalized_gold": n_gold,
            "normalized_pred": n_pred
        })
    if is_em:
        em_correct += 1
    total += 1

    db_path = os.path.join(db_dir, db_id, f"{db_id}.sqlite")
    gold_res = execute_sql(gold_sql, db_path)
    pred_res = execute_sql(pred_sql, db_path)
    if gold_res is not None and pred_res is not None:
        exec_total += 1
        if gold_res == pred_res:
            exec_matches += 1
        elif len(exec_mismatches) < 5:
            exec_mismatches.append({
                "question": question,
                "gold_sql": gold_sql,
                "pred_sql": pred_sql,
                "gold_result": gold_res,
                "pred_result": pred_res
            })

print("\n" + "-" * 45)
print(f"Exact Match   : {em_correct}/{total} = {em_correct/total*100:.2f}%")
if exec_total:
    print(f"Execution Accuracy : {exec_matches}/{exec_total} = {exec_matches/exec_total*100:.2f}%")
else:
    print("No queries ran without error.")

print("Exact-Match Mismatch Examples:")
for ex in em_mismatches:
    print(f"- Q: {ex['question']}")
    print(f"  Gold: {ex['normalized_gold']}")
    print(f"  Pred: {ex['normalized_pred']}\n")

print("Execution-Match Mismatch Examples:")
for ex in exec_mismatches:
    print(f"- Q: {ex['question']}")
    print(f"  Gold Result: {ex['gold_result']}")
    print(f"  Pred Result: {ex['pred_result']}\n")


100%|██████████| 1034/1034 [26:31<00:00,  1.54s/it]


---------------------------------------------
Exact Match   : 334/1034 = 32.30%
Execution Accuracy : 467/622 = 75.08%
Exact-Match Mismatch Examples:
- Q: Show name, country, age for all singers ordered by age from the oldest to the youngest.
  Gold: select name, country, age from singer order by age desc
  Pred: select name, country, age from singer order by age

- Q: What is the average, minimum, and maximum age for all French singers?
  Gold: select avg(age), min(age), max(age) from singer where country = 'france'
  Pred: select avg(age), min(age), max(age) from singer where country = 'french'

- Q: What are the names and release years for all the songs of the youngest singer?
  Gold: select song_name, song_release_year from singer order by age limit 1
  Pred: select song_name, song_release_year from singer order by age asc limit 1

- Q: What is the name and capacity for the stadium with highest average attendance?
  Gold: select name, capacity from stadium order by average desc lim